In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG ecommerce_catalog")

products = spark.table("silver.products")
orders = spark.table("silver.orders")

In [0]:
#Gold1: category-level sales trends
category_trends = (
    orders.filter("order_status = 'Delivered'")
    .join(products.select("product_id", "category", "brand"), "product_id", "left")
    .groupBy("category")
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    )
)
category_trends.write.format("delta").mode("overwrite").saveAsTable("gold.category_sales_trends")

In [0]:
#Gold2: Brand Performnace

brand_perf = (
    orders.filter("order_status = 'Delivered'")
    .join(products.select("product_id", "brand"), "product_id", "left")
    .groupBy("brand")
    .agg(
        F.count("*").alias("units_sold"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

brand_perf.write.format("delta").mode("overwrite").saveAsTable("gold.brand_performance")


In [0]:
#Gold 3: Monthly revenue summary

monthly_revenue = (
    orders.filter("order_status = 'Delivered'")
    .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("order_month", "payment_mode")
    .agg(F.count("*").alias("order_count"), F.round(F.sum("total_amount"), 2).alias("total_revenue"))
)
monthly_revenue.write.format("delta").mode("overwrite").saveAsTable("gold.monthly_revenue_summary")

In [0]:
#gold 4: Cutomer order summary (RFM-style base table)
customer_summary = (
    orders.filter("order_status = 'Delivered'")
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.max("order_date").alias("last_order_date")
        )
)

customer_summary.write.format("delta").mode("overwrite").saveAsTable("gold.customer_order_summary")

print("Gold Layer refreshed")